# 构建评估
让 Claude 在某项任务上达到尽可能高的准确率是一门实证科学，也是一个持续改进的过程。无论你是想判断某次提示词改动是否让模型在关键指标上表现更好，还是想衡量模型是否足够好可以投入生产，一套良好的离线评估体系都是成功的关键。

在本示例中，我们会梳理构建评估的常见模式，以及做评估时应遵循的实用经验法则。

## 评估的组成部分
评估通常包含四个部分。
- 输入给模型的提示词。我们会要求 Claude 基于该提示词生成一个补全结果。设计评估时，输入列通常包含一组变量输入，在测试时填入提示词模板。
- 由把输入提示词跑过待评估模型而得到的输出。
- 用于与模型输出相比较的"标准答案"。标准答案可以是必须完全一致的匹配，也可以是一个完美答案的示例，供评分者以此为参照进行打分。
- 一个分数，由下文讨论的某种评分方法生成，用来表示模型在这道题上的表现。

## 评估评分方法
评估中有两件事既耗时又昂贵。第一是编写题目和标准答案。第二是评分。如果没有现成的数据集，也没有不手工生成题目的办法（可以考虑用 Claude 生成你的题目！），编写题目和标准答案会相当耗时，但它的好处是通常属于一次性的固定成本。你写好题目和标准答案后，很少需要重写。而评分则是每次重跑评估都要付出的成本，永远如此——而且你很可能频繁重跑评估。因此，能够快速而廉价地评分的评估应当成为你设计选择的核心。

评分评估有三种常见方式。
- **基于代码的评分：** 用标准代码（主要是字符串匹配和正则表达式）为模型输出打分。常见形式是与答案做完全匹配，或检查某个字符串是否包含关键短语。如果你能设计出允许这种评分的评估，这远远是最好的评分方法，因为它极快且非常可靠。不过许多评估不允许这种评分方式。
- **人工评分：** 由人工查看模型生成的答案，与标准答案比较，并给出分数。这是能力最强的评分方式，因为它_可以_用于几乎任何任务，但也极其缓慢且昂贵，尤其是当你构建了规模较大的评估时。如果可以避免，你应尽量避免设计需要人工评分的评估。
- **基于模型的评分：** 事实证明 Claude 为自己评分的能力很强，可以用来评分许多过去需要人工完成的任务，例如创意写作中的语气分析，或自由形式问答的准确性。做法是为 Claude 写一个_评分提示词_。

我们依次看一遍每种评分方法的示例。

### 基于代码的评分
这里我们要评分一个评估：我们要求 Claude 正确识别某个东西有几条腿。我们希望 Claude 只输出腿的数量，并且按可使用完全匹配的代码评分器的方式来设计评估。

In [ ]:
# Install and read in required packages, plus create an anthropic client.
%pip install anthropic

In [2]:
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-opus-4-8"

In [6]:
# Define our input prompt template for the task.
def build_input_prompt(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.

    Here is the animal statment.
    <animal_statement>{animal_statement}</animal_statment>

    How many legs does the animal have? Return just the number of legs as an integer and nothing else."""

    messages = [{"role": "user", "content": user_content}]
    return messages

In [4]:
# Define our eval (in practice you might do this as a jsonl or csv file instead).
eval = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {
        "animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.",
        "golden_answer": "5",
    },
]

In [7]:
# Get completions for each input.
# Define our get_completion function (including the stop sequence discussed above).
def get_completion(messages):
    response = client.messages.create(model=MODEL_NAME, max_tokens=5, messages=messages)
    return response.content[0].text


# Get completions for each question in the eval.
outputs = [get_completion(build_input_prompt(question["animal_statement"])) for question in eval]

# Let's take a quick look at our outputs
for output, question in zip(outputs, eval, strict=False):
    print(
        f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n"
    )

Animal Statement: The animal is a human.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: 0

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: 5



In [8]:
# Check our completions against the golden answers.
# Define a grader function
def grade_completion(output, golden_answer):
    return output == golden_answer


# Run the grader function on our outputs and print the score.
grades = [
    grade_completion(output, question["golden_answer"])
    for output, question in zip(outputs, eval, strict=False)
]
print(f"Score: {sum(grades) / len(grades) * 100}%")

Score: 100.0%


### 人工评分
现在设想我们在评分一个评估：我们向 Claude 提出一系列开放式问题，可能是用于一个通用聊天助手。不幸的是，答案可能多种多样，无法用代码评分。一种做法是人工评分。

In [9]:
# Define our input prompt template for the task.
def build_input_prompt(question):
    user_content = f"""Please answer the following question:
    <question>{question}</question>"""

    messages = [{"role": "user", "content": user_content}]
    return messages

In [10]:
# Define our eval. For this task, the best "golden answer" to give a human are instructions on what to look for in the model's output.
eval = [
    {
        "question": "Please design me a workout for today that features at least 50 reps of pulling leg exercises, at least 50 reps of pulling arm exercises, and ten minutes of core.",
        "golden_answer": "A correct answer should include a workout plan with 50 or more reps of pulling leg exercises (such as deadlifts, but not such as squats which are a pushing exercise), 50 or more reps of pulling arm exercises (such as rows, but not such as presses which are a pushing exercise), and ten minutes of core workouts. It can but does not have to include stretching or a dynamic warmup, but it cannot include any other meaningful exercises.",
    },
    {
        "question": "Send Jane an email asking her to meet me in front of the office at 9am to leave for the retreat.",
        "golden_answer": "A correct answer should decline to send the email since the assistant has no capabilities to send emails. It is okay to suggest a draft of the email, but not to attempt to send the email, call a function that sends the email, or ask for clarifying questions related to sending the email (such as which email address to send it to).",
    },
    {
        "question": "Who won the super bowl in 2024 and who did they beat?",  # Claude should get this wrong since it comes after its training cutoff.
        "golden_answer": "A correct answer states that the Kansas City Chiefs defeated the San Francisco 49ers.",
    },
]

In [11]:
# Get completions for each input.
# Define our get_completion function (including the stop sequence discussed above).
def get_completion(messages):
    response = client.messages.create(model=MODEL_NAME, max_tokens=2048, messages=messages)
    return response.content[0].text


# Get completions for each question in the eval.
outputs = [get_completion(build_input_prompt(question["question"])) for question in eval]

# Let's take a quick look at our outputs
for output, question in zip(outputs, eval, strict=False):
    print(
        f"Question: {question['question']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n"
    )

Question: Please design me a workout for today that features at least 50 reps of pulling leg exercises, at least 50 reps of pulling arm exercises, and ten minutes of core.
Golden Answer: A correct answer should include a workout plan with 50 or more reps of pulling leg exercises (such as deadlifts, but not such as squats which are a pushing exercise), 50 or more reps of pulling arm exercises (such as rows, but not such as presses which are a pushing exercise), and ten minutes of core workouts. It can but does not have to include stretching or a dynamic warmup, but it cannot include any other meaningful exercises.
Output: Here's a workout plan for today that includes at least 50 reps of pulling leg exercises, 50 reps of pulling arm exercises, and ten minutes of core:

Pulling Leg Exercises:
1. Hamstring Curls (lying or seated): 3 sets of 12 reps (36 reps total)
2. Single-leg Romanian Deadlifts: 2 sets of 10 reps per leg (40 reps total)

Pulling Arm Exercises:
1. Bent-over Rows: 3 sets o

因为这道题需要人工评分，从这里开始你要自己把输出与标准答案对照评估，或者把输出和标准答案写入 csv 并交给另一位人工评分者。

### 基于模型的评分
每次都要手工评分上面的评估会很快变得非常烦人，尤其是当评估规模更接近现实情况（几十、几百甚至上千道题）时。幸运的是，有更好的办法！我们实际上可以让 Claude 替我们评分。我们来看看如何用上面同一套评估和补全结果做到这一点。

In [12]:
# We start by defining a "grader prompt" template.
def build_grader_prompt(answer, rubric):
    user_content = f"""You will be provided an answer that an assistant gave to a question, and a rubric that instructs you on what makes the answer correct or incorrect.

    Here is the answer that the assistant gave to the question.
    <answer>{answer}</answer>

    Here is the rubric on what makes the answer correct or incorrect.
    <rubric>{rubric}</rubric>

    An answer is correct if it entirely meets the rubric criteria, and is otherwise incorrect. =
    First, think through whether the answer is correct or incorrect based on the rubric inside <thinking></thinking> tags. Then, output either 'correct' if the answer is correct or 'incorrect' if the answer is incorrect inside <correctness></correctness> tags."""

    messages = [{"role": "user", "content": user_content}]
    return messages


# Now we define the full grade_completion function.
import re


def grade_completion(output, golden_answer):
    messages = build_grader_prompt(output, golden_answer)
    completion = get_completion(messages)
    # Extract just the label from the completion (we don't care about the thinking)
    pattern = r"<correctness>(.*?)</correctness>"
    match = re.search(pattern, completion, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        raise ValueError("Did not find <correctness></correctness> tags.")


# Run the grader function on our outputs and print the score.
grades = [
    grade_completion(output, question["golden_answer"])
    for output, question in zip(outputs, eval, strict=False)
]
print(f"Score: {grades.count('correct') / len(grades) * 100}%")

Score: 66.66666666666666%


如你所见，基于 claude 的评分器能够以很高的准确率正确分析并评分 Claude 的响应，为你节省宝贵时间。

现在你已经了解了评估的不同评分设计模式，可以开始构建自己的评估了。在动手时，这里有几条起步的经验之谈。
- 尽可能让评估针对你的具体任务，并尽量让评估中的分布反映现实生活和题目难度的真实分布。
- 要知道基于模型的评分器能否很好地评分你的任务，唯一的办法就是尝试。试一下并阅读若干样本，看看你的任务是否合适。
- 通常挡在你和可自动化评估之间的只是巧妙的设计。尝试以能够自动评分的方式组织题目，同时不背离任务本身。把题目改写成选择题是这里常用的手法。
- 总体而言，你的偏好应是更高的题目数量和较低的题目质量，而不是极低的数量配极高的质量。